# 04 - Evaluation & Explainability

Consolidated evaluation of the best model (`improved_opt`, Ghost+MPCA+SIoU) on the held-out
**TEST** split (180 imgs, never seen in training/selection). Covers: overall + per-class
metrics, confusion matrix / PR curve, qualitative predictions, and **Eigen-CAM** explanations.

Run `01_data_preparation.ipynb` first, and train at least `updated_05_train_improved.ipynb`.

## 1. Setup (register custom modules so the improved checkpoint loads)

In [1]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))
from src.modules import register, register_lzy
register(); register_lzy()                      # so improved / LZY best.pt un-pickle

from ultralytics import YOLO
DATA_CFG = ROOT / 'data' / 'neu-det-yolo' / 'data.yaml'
CLASSES = ['crazing','inclusion','patches','pitted_surface','rolled-in_scale','scratches']
BEST = ROOT / 'results' / 'improved_opt' / 'weights' / 'best.pt'
assert DATA_CFG.exists(), 'Run 01_data_preparation.ipynb first!'
assert BEST.exists(), 'Train updated_05_train_improved.ipynb first!'
print('best model:', BEST)

[src.modules.register] activated: MPCA (end-of-backbone attention), SIoU regression loss
[src.modules.register_lzy] activated: ResBlock_CBAM (head attention), WIoU regression loss
best model: c:\Users\student\Desktop\SteelDefectDetection\results\improved_opt\weights\best.pt


## 2. Test-set metrics (paper-comparable, with TTA + NMS 0.6)

In [2]:
model = YOLO(str(BEST))
tm = model.val(data=str(DATA_CFG), split='test', augment=True, iou=0.6, verbose=False)
print('--- TEST set (180 imgs) - improved_opt ---')
print('mAP@0.5      :', round(float(tm.box.map50), 4), '  (paper improved target: 0.786)')
print('mAP@0.5:0.95 :', round(float(tm.box.map), 4))
print('precision    :', round(float(tm.box.mp), 4))
print('recall       :', round(float(tm.box.mr), 4))

Ultralytics 8.4.51  Python-3.10.8 torch-2.6.0+cu124 CUDA:0 (NVIDIA RTX 2000 Ada Generation, 16380MiB)
YOLOv8n_improved summary (fused): 138 layers, 2,391,462 parameters, 0 gradients, 6.2 GFLOPs
val: Fast image access  (ping: 0.00.0 ms, read: 54.516.7 MB/s, size: 14.7 KB)
val: Scanning C:\Users\student\Desktop\SteelDefectDetection\data\neu-det-yolo\labels\test.cache... 180 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 180/180  0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 12/12 1.4it/s 8.4s0.7s
                   all        180        413      0.702      0.681      0.736      0.365
Speed: 4.2ms preprocess, 26.9ms inference, 0.0ms loss, 3.2ms postprocess per image
Results saved to C:\Users\student\Desktop\SteelDefectDetection\notebooks\runs\detect\val-37
--- TEST set (180 imgs) - improved_opt ---
mAP@0.5      : 0.7361   (paper improved target: 0.786)
mAP@0.5:0.95 : 0.3652
precision    : 0.7018
recall       : 0.6808


## 3. Per-class mAP@0.5 (worst -> best)

In [3]:
rows = [{'class': n, 'mAP@0.5': round(float(tm.box.ap50[i]), 4),
         'mAP@0.5:0.95': round(float(tm.box.ap[i]), 4)} for i, n in enumerate(CLASSES)]
df = pd.DataFrame(rows).sort_values('mAP@0.5').reset_index(drop=True)
display(df)
ax = df.plot.barh(x='class', y='mAP@0.5', legend=False, color='#1f5f8b', figsize=(8, 4))
ax.set_title('Per-class mAP@0.5  (improved_opt, TEST)'); ax.set_xlim(0, 1)
plt.tight_layout(); plt.show()

,class,mAP@0.5,mAP@0.5:0.95
0,crazing,0.4486,0.1836
1,rolled-in_scale,0.5469,0.2134
2,inclusion,0.8180,0.4217
3,scratches,0.8351,0.3700
4,pitted_surface,0.8540,0.4206
5,patches,0.9139,0.5817


<Figure size 800x400 with 1 Axes>

## 4. Confusion matrix & PR curve (saved during training)

In [4]:
run = ROOT / 'results' / 'improved_opt'
for name in ['confusion_matrix.png', 'BoxPR_curve.png']:
    p = run / name
    if p.exists():
        plt.figure(figsize=(7, 6)); plt.imshow(Image.open(p)); plt.axis('off')
        plt.title(name); plt.show()
    else:
        print('missing (re-run training with plots=True):', p)

<Figure size 700x600 with 1 Axes>

<Figure size 700x600 with 1 Axes>

## 5. Qualitative predictions - one TEST sample per class

In [5]:
import random
random.seed(0)
test_imgs = sorted((ROOT / 'data' / 'neu-det-yolo' / 'images' / 'test').glob('*.jpg'))
samples = []
for c in CLASSES:
    cimgs = [p for p in test_imgs if p.name.startswith(c + '_')]
    if cimgs:
        samples.append(random.choice(cimgs))

fig, axes = plt.subplots(2, 3, figsize=(13, 8))
for ax, img_path in zip(axes.flat, samples):
    r = model.predict(str(img_path), conf=0.25, verbose=False)[0]
    ax.imshow(r.plot()[:, :, ::-1]); ax.set_title(img_path.name.split('_')[0]); ax.axis('off')
plt.suptitle('improved_opt - sample TEST predictions', fontsize=14)
plt.tight_layout(); plt.show()

<Figure size 1300x800 with 6 Axes>

## 6. Eigen-CAM explanations - where the model looks
Bright regions drive the detector. Watch **crazing** and **rolled-in_scale**: the heatmaps go
diffuse (no localized texture to lock onto), which visually explains their lower mAP@0.5 -
the signal is genuinely weak, the model is not simply failing.

In [6]:
from src.explain import EigenCAM, overlay_cam
cam = EigenCAM(model, device='cpu')
fig, axes = plt.subplots(2, 3, figsize=(13, 8))
for ax, img_path in zip(axes.flat, samples):
    im = Image.open(img_path).convert('RGB')
    heat = cam(im, imgsz=640)
    ax.imshow(overlay_cam(im, heat)); ax.set_title(img_path.name.split('_')[0]); ax.axis('off')
cam.close()
plt.suptitle('Eigen-CAM - improved_opt (where the detector looks)', fontsize=14)
plt.tight_layout(); plt.show()

<Figure size 1300x800 with 6 Axes>

## 7. Model comparison (all runs)

In [7]:
cmp = ROOT / 'results' / 'model_comparison.txt'
print(cmp.read_text() if cmp.exists() else 'results/model_comparison.txt not found')

Steel Surface Defect Detection - Model Comparison (TEST set, 180 imgs)

                              params(M)  mAP@0.5  mAP@0.5:0.95  precision  recall  paper/repo target
model                                                                                               
baseline (YOLOv8n)                 3.01   0.7367        0.3823     0.6999  0.6826              0.774
paper (Ghost+MPCA+SIoU)            2.39   0.7073        0.3606     0.6661  0.6362              0.786
LZY-233 (Ghost+ResCBAM+WIoU)       4.05   0.7316        0.3685     0.6412  0.7320              0.792

Per-class mAP@0.5:
                 baseline (YOLOv8n)  paper (Ghost+MPCA+SIoU)  LZY-233 (Ghost+ResCBAM+WIoU)
class (mAP@0.5)                                                                           
crazing                      0.4402                   0.4595                        0.4438
inclusion                    0.8386                   0.7986                        0.8039
patches                      0.9315    

Evaluation complete. The best model (`improved_opt`) and its Eigen-CAM overlays are what the
Streamlit app (`src/app.py`) and the Hugging Face Space serve.